# Minimal example: from a Kumu Excel to an optimized intervention

This tutorial walks through the full Systems Insight Pipeline (SIP) workflow on the
smallest possible model, [`Minimal.xlsx`](Minimal.xlsx). It is the best place to start and
the best template to copy when building your own model.

## The Excel file

A model is a causal loop diagram (CLD) described in an Excel workbook with the same layout
Kumu (kumu.io) uses for its exports, so you can draw your diagram in Kumu, export it, and
annotate the export — or write the workbook directly.

**Elements sheet** — one row per variable:

| Label | Type | Tags | Description | Equation |
|-------|------|------|-------------|----------|
| A | auxiliary | 0 | VOI | `#1*B*C+0.01*#2*B+0.01*#3*C` |
| B | constant | 1 | | |
| C | constant | 1 | | |

- **Type**: `stock` (accumulates over time), `auxiliary` (instantaneous function of other
  variables), or `constant` (exogenous).
- **Tags**: a nonzero value marks the variable as an *intervention* with that relative
  strength (sign sets the direction).
- **Description**: `VOI` marks the *variable of interest* — the outcome to analyze.
- **Equation** (optional): a custom equation overriding the default linear combination of
  incoming links. `#1`, `#2`, ... are free parameters sampled uniformly from
  `[0, parameter_value_aux]`; `$` marks where an intervention enters (appended if absent);
  numpy functions like `np.exp`, `np.tanh` are available. Here `A` responds to the
  *product* of `B` and `C` plus small linear terms.

**Connections sheet** — one row per causal link, with `From`, `To`, and `Type` (`+` or `-`
polarity): `B → A (+)` and `C → A (+)`.

An optional **Interactions sheet** can declare pairwise interaction terms
(`From1`, `From2`, `To`, `Type`) for models without custom equations.

In [ ]:
from sip_systemsinsightpipeline import Extract, SDM, SDMOptimizer
from sip_systemsinsightpipeline.plots import plot_simulated_intervention_ranking

extract = Extract("Minimal.xlsx")
s = extract.extract_settings()

# Simulation settings
s.seed = 12345                  # Reproducibility
s.N = 50                        # Number of parameter samples
s.t_end = 10                    # Time horizon
s.time_unit = "years"
s.parameter_value_aux = 0.1     # Upper bound for sampled auxiliary/equation parameters
s.parameter_value_stocks = 0.1  # Upper bound for sampled stock parameters

sdm = SDM(s)

## Simulate interventions under parameter uncertainty

Every link coefficient (and every `#` parameter of a custom equation) is unknown, so SIP
samples them `N` times and simulates each tagged intervention for every sample. The ranking
plot shows the distribution of each intervention's effect on the variable of interest at
the final time point.

In [ ]:
df_sol_per_sample, param_samples = sdm.run_simulations()
intervention_effects = sdm.get_intervention_effects()

voi = s.variable_of_interest[0]
fig = plot_simulated_intervention_ranking(s, intervention_effects[voi], voi)

## Optimize a budget split between interventions

Instead of testing one intervention at a time, the optimizer allocates a **budget** across
all interventions. It works in expenditure space: each intervention gets an expenditure
`y_i = cost_i * intensity_i`, subject to `y_i >= 0` and `sum(y) <= budget`. Multi-start
SLSQP (Sobol-seeded) finds the allocation that maximizes (or minimizes) the variable of
interest, and reports near-optimal alternatives ("equilibria") when several allocations
perform comparably.

In [ ]:
params = sdm.sample_model_parameters()  # one parameter draw
costs = [1.0] * len(s.intervention_variables)

optimizer = SDMOptimizer(sdm)
result = optimizer.optimize_intervention_intensities(
    params=params,
    costs=costs,
    variable_of_interest=voi,
    budget=1.0,
    maximize=True,
    n_starts=8,
    seed=1,
)

print("Success:", result["success"])
print("Best effect size:", round(result["best_effect_size"], 5))
print("Near-optimal allocations found:", result["n_equilibria"])
for var, intensity in zip(s.intervention_variables, result["best_intensities"]):
    print(f"  {var}: intensity {intensity:.3f}")

Because `A`'s equation rewards the *product* `B*C`, the optimal allocation splits the
budget roughly evenly between the two interventions instead of spending it all on one —
something a one-intervention-at-a-time ranking cannot reveal.

## How robust is the optimal allocation?

Re-running the optimization for many parameter draws shows how the optimal allocation
shifts with parameter uncertainty. The result is a tidy DataFrame with one row per
near-optimal allocation per parameter sample.

In [ ]:
opt_df = optimizer.optimize_across_parameter_samples(
    costs=costs,
    variable_of_interest=voi,
    maximize=True,
    n_starts=4,
    seed=1,
)

intensity_cols = [c for c in opt_df.columns if c.startswith("intensity_")]
print(f"{len(opt_df)} allocations across {opt_df['sample_idx'].nunique()} parameter samples")
print("\nMean optimal intensity per intervention:")
print(opt_df[intensity_cols].mean().round(3).to_string())
opt_df.head()

## Next steps

- [`Insulation.ipynb`](Insulation.ipynb) applies the same workflow to a real-world model of
  household energy decisions, including sensitivity analysis and feedback-loop dominance
  (Loops That Matter).
- The [README](../README.md) documents the Excel format and the API in more detail.